In [1]:
#Translator using Azure API
#Install packages
!pip install requests uuid


In [ ]:
#Simple translator
import os, uuid, json, requests

subscription_key = '<key>'
endpoint = 'https://api.cognitive.microsofttranslator.com'

url = f"{endpoint}/translate"
to_lang = "pt-BR"
params = {"api-version": "3.0", "to": to_lang}
region = "eastus"

headers = {
    "Ocp-Apim-Subscription-Key": subscription_key,
    "Ocp-Apim-Subscription-Region": region,
    "Content-type": "application/json",
    "X-ClientTraceId": str(uuid.uuid4())
}

body = [{"text": "Hello, world!"}]

resp = requests.post(url, params=params, headers=headers, json=body, timeout=15)

if not resp.ok:
    raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")

data = resp.json()
print(json.dumps(data, ensure_ascii=False, indent=2))
print("Translated:", data[0]["translations"][0]["text"])

[
  {
    "detectedLanguage": {
      "language": "en",
      "score": 0.9
    },
    "translations": [
      {
        "text": "Olá, mundo!",
        "to": "pt"
      }
    ]
  }
]
Translated: Olá, mundo!


In [2]:
!pip -q install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.5 MB/s eta 0:00:00


In [3]:
#Docx Translator
import os, uuid, json, requests
from docx import Document
from google.colab import files
uploaded = files.upload()
in_path = next(iter(uploaded.keys()))
in_path

subscription_key = '<key>'
endpoint = "https://api.cognitive.microsofttranslator.com"
url = f"{endpoint}/translate"

to_lang = "pt-BR"
region = "eastus"
params = {"api-version": "3.0", "to": to_lang}

headers = {
    "Ocp-Apim-Subscription-Key": subscription_key,
    "Ocp-Apim-Subscription-Region": region,
    "Content-type": "application/json",
    "X-ClientTraceId": str(uuid.uuid4()),
}

def translate_batch(texts):
    """Translate a list of strings; returns list same length."""
    body = [{"text": t} for t in texts]
    resp = requests.post(url, params=params, headers=headers, json=body, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
    data = resp.json()
    return [item["translations"][0]["text"] for item in data]

def iter_paragraphs_in_doc(doc):
    """Yield all paragraph objects in body and inside tables."""
    for p in doc.paragraphs:
        yield p
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    yield p

def translate_docx(in_docx, out_docx, chunk_size=80):
    doc = Document(in_docx)

    paras = list(iter_paragraphs_in_doc(doc))
    texts = [p.text for p in paras]

    # Keep empties unchanged; translate only non-empty
    idxs = [i for i,t in enumerate(texts) if t and t.strip()]
    if not idxs:
        doc.save(out_docx)
        return

    nonempty = [texts[i] for i in idxs]

    translated = []
    for start in range(0, len(nonempty), chunk_size):
        translated.extend(translate_batch(nonempty[start:start+chunk_size]))

    # Put translated text back
    for k, i in enumerate(idxs):
        paras[i].text = translated[k]

    doc.save(out_docx)

base = os.path.splitext(os.path.basename(in_path))[0]
out_path = f"{base}_translated_{to_lang}.docx"
translate_docx(in_path, out_path)
out_path

files.download(out_path)

Saving music_test_translation.docx to music_test_translation.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
out_path

'music_test_translation_translated_pt-BR.docx'

In [ ]:
!pip -q install beautifulsoup4 readability-lxml markdownify

In [ ]:
#Translate URL and output as markdown
import os, uuid, requests
from bs4 import BeautifulSoup
from readability import Document as ReadabilityDoc
from markdownify import markdownify as md

subscription_key = '<key>'
endpoint = "https://api.cognitive.microsofttranslator.com"
url_translator = f"{endpoint}/translate"

to_lang = "pt-BR"
region = "eastus"
params = {"api-version": "3.0", "to": to_lang}

headers = {
    "Ocp-Apim-Subscription-Key": subscription_key,
    "Ocp-Apim-Subscription-Region": region,
    "Content-type": "application/json",
    "X-ClientTraceId": str(uuid.uuid4()),
}

def fetch_html(url: str, timeout=20) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; colab-translator/1.0)",
        "Accept": "text/html,application/xhtml+xml",
    }
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.text

def extract_main_html(html: str) -> str:
    """
    Best-effort: tries Readability to isolate main article content.
    Falls back to full page if extraction fails.
    """
    try:
        doc = ReadabilityDoc(html)
        return doc.summary(html_partial=True)
    except Exception:
        return html

def html_to_markdown(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Remove junk
    for tag in soup(["script", "style", "noscript", "iframe", "svg"]):
        tag.decompose()

    cleaned_html = str(soup)
    # Convert to markdown
    return md(cleaned_html, heading_style="ATX").strip()

def split_for_translation(text: str, max_chars=3500):
    """
    Splits markdown into chunks under max_chars.
    Character-based split to keep it simple and robust.
    """
    chunks, buf = [], []
    size = 0
    for line in text.splitlines(True):  # keep line breaks
        if size + len(line) > max_chars and buf:
            chunks.append("".join(buf))
            buf, size = [], 0
        buf.append(line)
        size += len(line)
    if buf:
        chunks.append("".join(buf))
    return chunks

def translate_chunks(chunks, to_lang="pt-BR", region="eastus"):
    out = []
    # Translator accepts a list; keep batch size modest
    batch_size = 30
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        body = [{"text": c} for c in batch]
        resp = requests.post(url_translator, params=params, headers=headers, json=body, timeout=30)
        if not resp.ok:
            raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
        data = resp.json()
        out.extend([item["translations"][0]["text"] for item in data])
    return out

def scrape_translate_markdown(url: str, to_lang="pt-BR"):
    html = fetch_html(url)
    main_html = extract_main_html(html)
    markdown = html_to_markdown(main_html)

    chunks = split_for_translation(markdown, max_chars=3500)
    translated_chunks = translate_chunks(chunks, to_lang=to_lang, region=region)

    translated_md = "".join(translated_chunks).strip()
    return markdown, translated_md

# ---- Use it ----
url = input("Paste URL: ").strip()
orig_md, pt_md = scrape_translate_markdown(url, to_lang="pt-BR")

print("=== TRANSLATED MARKDOWN ===\n")
print(pt_md)

# Optional: save
with open("translated.md", "w", encoding="utf-8") as f:
    f.write(pt_md)
print("\nSaved: translated.md")



Paste URL: https://finance.yahoo.com/news/why-ibm-ibm-stock-down-165116217.html
=== TRANSLATED MARKDOWN ===

As ações da gigante de tecnologia e consultoria IBM (NYSE:IBM) caíram 4% na sessão da manhã após a Anthropic lançar uma nova ferramenta de IA dentro do Claude Code, projetada para automatizar a modernização do COBOL, uma linguagem de programação legada que alimenta a espinha dorsal das finanças globais e das companhias aéreas.  
  
Esse desenvolvimento ameaça diretamente os serviços de consultoria e infraestrutura de alta margem da IBM, que tradicionalmente dependem de atualizações manuais e legadas de vários anos.  
  
Para aumentar a preocupação, o anúncio de novas tarifas globais pela administração Trump reacendeu a incerteza na política comercial. A medida ocorreu rapidamente após a Suprema Corte decidir, na semana anterior, que o presidente não poderia usar a Lei Internacional de Poderes Econômicos de Emergência (IEEPA) para tais funções, uma decisão que inicialmente havia 